# CNN completa

**Capítulo 4 · Universidad de las Hespérides**

Adaptación al español de *Dive into Deep Learning*, Aston Zhang, Zachary C. Lipton, Mu Li y Alexander J. Smola.
Fuente: `locked/chapter_convolutional-neural-networks/lenet.ipynb` · [Lección original](https://d2l.ai/chapter_convolutional-neural-networks/lenet.html).
Texto adaptado bajo [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/). [Procedencia y cambios](../PROCEDENCIA.md).
Se conserva la secuencia de las celdas y de los ejercicios; las notas de Hespérides se identifican expresamente.

**Entorno:** ejecuta `uv sync` en la raíz y selecciona su Python como kernel. Las descargas se realizan una vez y quedan en `data/`.
Por defecto, el soporte limita los entrenamientos de `Trainer` a tres épocas y 1024/256 ejemplos para CPU.
Para repetir el régimen completo, inicia Jupyter con `HESPERIDES_COMPLETO=1`. Los ejemplos visuales pequeños conservan su propia configuración explícita.
Los datos de texto en inglés o francés son entradas de los experimentos originales y mantienen su idioma.


In [ ]:
from pathlib import Path
import sys
RAIZ = Path.cwd() if (Path.cwd() / "laboratorio").exists() else Path.cwd().parent
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))
from laboratorio import d2l, configurar, epocas
configurar()


# Redes neuronales convolucionales (LeNet)
<a id="sec_lenet"></a>

Ahora tenemos todos los ingredientes necesarios para ensamblar una CNN completamente funcional. En nuestro encuentro anterior con datos de imagen, aplicamos un modelo lineal con regresión softmax ([Referencia sec_softmax_scratch](https://d2l.ai/chapter_linear-classification/softmax-regression-scratch.html#sec-softmax-scratch)) y un MLP ([Referencia sec_mlp-implementation](https://d2l.ai/chapter_multilayer-perceptrons/mlp-implementation.html#sec-mlp-implementation)) a imágenes de ropa en el conjunto de datos de la moda-MNIST. Para hacer que tales datos sean fáciles primero aplanamos cada imagen de una matriz $28\times28$ en un vector $784$-dimensional de longitud fija, y después las procesamos en capas totalmente conectadas. Ahora que tenemos un mango en capas convolucionales, podemos retener la estructura espacial en nuestras imágenes. Como un beneficio adicional de reemplazar capas totalmente conectadas con capas convolucionales, disfrutaremos de modelos más parsimoniosos que requieren muchos menos parámetros.

En esta sección, presentaremos *LeNet*, entre los primeros CNNs publicados para captar una amplia atención por su desempeño en tareas de visión computarizada. El modelo fue introducido por (y nombrado por) Yann LeCun, entonces investigador en AT&T Bell Labs, con el propósito de reconocer dígitos escritos a mano en imágenes [LeCun.Bottou.Bengio.ea.1998](https://d2l.ai/chapter_references/zreferences.html). Este trabajo representó la culminación de una década de investigación desarrollando la tecnología; el equipo de LeCun publicó el primer estudio para entrenar exitosamente CNNs a través de backpropagation [LeCun.Boser.Denker.ea.1989](https://d2l.ai/chapter_references/zreferences.html).

En el momento en que LeNet logró resultados excepcionales que coinciden con el rendimiento de las máquinas vectoriales de soporte, entonces un enfoque dominante en el aprendizaje supervisado, logrando una tasa de error de menos de 1% por dígito. LeNet fue finalmente adaptado para reconocer dígitos para procesar depósitos en máquinas ATM. Hasta el día de hoy, algunos cajeros todavía funcionan el código que Yann LeCun y su colega Leon Bottou escribió en la década de 1990!


In [ ]:
import torch
from torch import nn
from laboratorio import d2l

## LeNet
En un nivel alto, **LeNet (LeNet-5) consta de dos partes: (i) un codificador convolucional que consta de dos capas convolucionales; y (ii) un bloque denso que consta de tres capas totalmente conectadas**. La arquitectura se resume en [Referencia img_lenet](https://d2l.ai/chapter_convolutional-neural-networks/lenet.html#img-lenet).

![Flujo de datos en LeNet: de un dígito manuscrito a una distribución sobre diez clases.](../recursos/originales/lenet.svg)
<a id="img_lenet"></a>

Las unidades básicas de cada bloque convolucional son una capa convolucional, una función de activación sigmoidea y una posterior operación de pooling media. Tenga en cuenta que si bien ReLUs y max-pooling funcionan mejor, todavía no se habían descubierto. Cada capa convolucional utiliza un núcleo $5\times 5$ y una función de activación sigmoidea. Estas capas mapean las entradas dispuestas espacialmente a un número de mapas de características bidimensionales, típicamente aumentando el número de canales. La primera capa convolucional tiene 6 canales de salida, mientras que la segunda tiene 16. Cada operación de pooling $2\times2$ (stride 2) reduce la dimensión mediante un factor de $4$ mediante toma de muestras espacial. El bloque convolucional emite una salida con forma dada por (tamaño de lote, número de canal, altura, anchura).

Para pasar la salida del bloque convolucional al bloque denso, debemos aplanar cada ejemplo en el minibatch. En otras palabras, tomamos esta entrada cuatridimensional y la transformamos en la entrada bidimensional esperada por capas totalmente conectadas: como recordatorio, la representación bidimensional que deseamos utiliza la primera dimensión para indexar ejemplos en el minibatch y la segunda para dar la representación vectorial plana de cada ejemplo. El bloque denso de LeNet tiene tres capas totalmente conectadas, con 120, 84 y 10 salidas, respectivamente. Debido a que todavía estamos realizando la clasificación, la capa de salida de 10 dimensiones corresponde al número de clases de salida posibles.

Al llegar al punto en el que usted realmente entiende lo que está pasando dentro de LeNet puede haber tomado un poco de trabajo, esperamos que el siguiente fragmento de código le convencerá de que la implementación de tales modelos con bibliotecas modernas de aprendizaje profundo es notablemente simple. Sólo necesitamos instanciar un bloque `Sequential` y encadenar las capas apropiadas, utilizando la inicialización Xavier como se introdujo en [Referencia subsec_xavier](https://d2l.ai/chapter_multilayer-perceptrons/numerical-stability-and-init.html#subsec-xavier).


In [ ]:
def init_cnn(module):  #@save
    """Inicialice pesos para CNNs."""
    if type(module) == nn.Linear or type(module) == nn.Conv2d:
        nn.init.xavier_uniform_(module.weight)

In [ ]:
class LeNet(d2l.Classifier):  #@save
    """El modelo LeNet-5."""
    def __init__(self, lr=0.1, num_classes=10):
        super().__init__()
        self.save_hyperparameters()
        self.net = nn.Sequential(
            nn.LazyConv2d(6, kernel_size=5, padding=2), nn.Sigmoid(),
            nn.AvgPool2d(kernel_size=2, stride=2),
            nn.LazyConv2d(16, kernel_size=5), nn.Sigmoid(),
            nn.AvgPool2d(kernel_size=2, stride=2),
            nn.Flatten(),
            nn.LazyLinear(120), nn.Sigmoid(),
            nn.LazyLinear(84), nn.Sigmoid(),
            nn.LazyLinear(num_classes))

Nos hemos tomado cierta libertad en la reproducción de LeNet en la medida en que hemos reemplazado la capa de activación gaussiana por una capa softmax. Esto simplifica en gran medida la implementación, sobre todo debido al hecho de que el decodificador gaussiano se utiliza raramente hoy en día. Aparte de eso, esta red coincide con la arquitectura original de LeNet-5.


### Nota docente de Hespérides

Comprueba primero las formas y el supuesto arquitectónico: localidad, compartición de pesos o conexión residual. El explorador permite seguir ventana, multiplicaciones y suma. En PyTorch, Conv2d implementa correlación cruzada; en aprendizaje profundo se suele llamar convolución a esta operación. El autoencoder 91 amplía el patrón MLP con reconstrucción; VAE se trata como contraste conceptual, al no existir un original válido en las fuentes locales.

Vínculo con los apuntes: sesión 4, «CNN completa».


Veamos qué sucede dentro de la red. Al pasar una imagen $28 \times 28$ de un solo canal (blanco y negro) a través de la red e imprimir la forma de salida en cada capa, podemos **inspeccionar el modelo** para asegurarnos de que sus operaciones se alinean con lo que esperamos de [Referencia img_lenet_vert](https://d2l.ai/chapter_convolutional-neural-networks/lenet.html#img-lenet-vert).


![Notación compacta de LeNet-5.](../recursos/originales/lenet-vert.svg)
<a id="img_lenet_vert"></a>


In [ ]:
@d2l.add_to_class(d2l.Classifier)  #@save
def layer_summary(self, X_shape):
    X = torch.randn(*X_shape)
    for layer in self.net:
        X = layer(X)
        print(layer.__class__.__name__, 'output shape:\t', X.shape)

model = LeNet()
model.layer_summary((1, 1, 28, 28))

Tenga en cuenta que la altura y la anchura de la representación en cada capa a lo largo del bloque convolucional se reduce (en comparación con la capa anterior). La primera capa convolucional utiliza dos píxeles de relleno para compensar la reducción de altura y anchura que de otro modo resultaría de usar un núcleo $5 \times 5$. Aparte, el tamaño de imagen de los píxeles $28 \times 28$ en el conjunto de datos OCR MNIST original es el resultado de *cortar* dos filas de píxeles (y columnas) de los escaneos originales que midieron los píxeles $32 \times 32$. Esto se hizo principalmente para ahorrar espacio (una reducción del 30%) en un momento en que importó megabytes.

Por el contrario, la segunda capa convolucional se olvida del relleno, y por lo tanto la altura y la anchura se reducen en cuatro píxeles. A medida que subimos la pila de capas, el número de canales aumenta la capa-sobre-capa de 1 en la entrada a 6 después de la primera capa convolucional y 16 después de la segunda capa convolucional. Sin embargo, cada capa de pooling reduce la altura y la anchura. Finalmente, cada capa totalmente conectada reduce la dimensión, emitiendo finalmente una salida cuya dimensión coincide con el número de clases.

## Entrenamiento
Ahora que hemos implementado el modelo, vamos a realizar un experimento para ver cómo el modelo LeNet-5 va en Fashion-MNIST**.

Aunque las CNN tienen menos parámetros, todavía pueden ser más caras de calcular que las MLPs similares porque cada parámetro participa en muchas más multiplicaciones. Si tiene acceso a una GPU, este podría ser un buen momento para ponerlo en acción para acelerar el entrenamiento. Tenga en cuenta que la clase `d2l.Trainer` se encarga de todos los detalles. Por defecto, inicializa los parámetros del modelo en los dispositivos disponibles. Al igual que con las MLPs, nuestra función de pérdida es la entropía cruzada, y la minimizamos a través de descenso por gradiente estocástico minibatch.


In [ ]:
trainer = d2l.Trainer(max_epochs=10, num_gpus=1)
data = d2l.FashionMNIST(batch_size=128)
model = LeNet(lr=0.1)
model.apply_init([next(iter(data.get_dataloader(True)))[0]], init_cnn)
trainer.fit(model, data)

## Resumen
Hemos hecho progresos significativos en este capítulo. Nos hemos trasladado de los MLP de los años 80 a los CNN de los años 90 y principios de los 2000. Las arquitecturas propuestas, por ejemplo, en la forma de LeNet-5 siguen siendo significativas, incluso hasta el día de hoy. Vale la pena comparar las tasas de error en Fashion-MNIST alcanzable con LeNet-5 tanto a la mejor posible con MLPs ([Referencia sec_mlp-implementation](https://d2l.ai/chapter_multilayer-perceptrons/mlp-implementation.html#sec-mlp-implementation)) y aquellos con arquitecturas significativamente más avanzadas como ResNet ([Referencia sec_resnet](https://d2l.ai/chapter_convolutional-modern/resnet.html#sec-resnet)). LeNet es mucho más similar a la última que a la primera. Una de las diferencias primarias, como veremos, es que mayores cantidades de computación permitieron arquitecturas significativamente más complejas.

Una segunda diferencia es la relativa facilidad con la que pudimos implementar LeNet. Lo que solía ser un desafío de ingeniería que valía meses de C++ y código de montaje, ingeniería para mejorar SN, una herramienta de aprendizaje profundo basada en Lisp [Bottou.Le-Cun.1988](https://d2l.ai/chapter_references/zreferences.html), y finalmente la experimentación con modelos ahora se puede lograr en minutos. Es este increíble impulso de productividad que ha democratizado enormemente el desarrollo de modelos de aprendizaje profundo. En el siguiente capítulo viajaremos por este conejo para ver dónde nos lleva.

## Ejercicios
1. Vamos a modernizar LeNet. Implemente y pruebe los siguientes cambios:
    1. Reemplazar el pooling promedio con max-pooling.
    1. Reemplazar la capa softmax con ReLU.
1. Trate de cambiar el tamaño de la red de estilo LeNet para mejorar su precisión, además de max-pooling y ReLU.
    1. Ajustar el tamaño de la ventana de convolución.
    1. Ajuste el número de canales de salida.
    1. Ajustar el número de capas de convolución.
    1. Ajuste el número de capas totalmente conectadas.
    1. Ajustar las tasas de aprendizaje y otros detalles de la formación (por ejemplo, inicialización y número de épocas).
1. Pruebe la red mejorada en el conjunto de datos MNIST original.
1. Mostrar las activaciones de la primera y segunda capa de LeNet para diferentes entradas (por ejemplo, suéteres y abrigos).
1. ¿Qué sucede con las activaciones cuando se alimentan imágenes significativamente diferentes en la red (por ejemplo, gatos, coches, o incluso ruido aleatorio)?


**Errata del original:** el ejercicio de modernización menciona sustituir softmax por ReLU. En esta arquitectura las activaciones ocultas son sigmoid; el clasificador entrega logits. Interpreta y justifica la corrección sin sustituir la salida de clasificación por ReLU.

[Debate del original](https://discuss.d2l.ai/t/74)
